# 04 — Parseo LLM de Criterios de Elegibilidad

Procesa dos tipos de casos del notebook 03:
- `llm_pending`: textos sin headers reconocibles — extracción completa por LLM
- `partial`: regex encontró solo una sección — LLM completa la sección faltante

Usa Llama 3.2 3B vía Ollama corriendo localmente.

**Input:** `data/03_eligibility_parsed.parquet` + `data/clinical_trials.parquet`  
**Output:** `data/04_eligibility_parsed_llm.parquet` (unión regex + LLM)

**`RERUN_ONLY = True`** → solo re-corre los `llm_partial` con ambas listas vacías del output existente.  
**`RERUN_ONLY = False`** → pipeline completo desde regex.

## 4.1 Configuration

In [18]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import requests
import json
import re
import time
import sys

PARQUET_REGEX      = Path('..') / 'data' / '03_eligibility_parsed.parquet'
PARQUET_RAW        = Path('..') / 'data' / 'clinical_trials.parquet'
PARQUET_OUTPUT     = Path('..') / 'data' / '04_eligibility_parsed_llm.parquet'
PARQUET_CHECKPOINT = Path('..') / 'data' / '04_eligibility_llm_checkpoint.parquet'

OLLAMA_URL        = 'http://localhost:11434'
MODEL             = 'llama3.2:3b-instruct-q4_K_M'
MAX_TOKENS        = 1500
MAX_RETRIES       = 3
N_WORKERS         = 4
PROGRESS_EVERY    = 25
CHECKPOINT_EVERY  = 100
REQUEST_TIMEOUT   = 60

# True  → solo re-corre llm_partial vacíos del output existente
# False → pipeline completo (regex pendientes + re-run)
RERUN_ONLY = True

TASK_ORDER = {'full': 0, 'missing_exclusion': 1, 'missing_inclusion': 2}

# Señales para detección previa de sección — evita alucinaciones
_EXCL_SIGNALS = re.compile(r'(?i)exclud|not eligible|must not|prohibited|contraindic|ineligible|not allowed')
_INCL_SIGNALS = re.compile(r'(?i)must have|required|eligible|confirmed|diagnosis|patients with|subjects with')

## 4.2 Load cases to process

In [19]:
df_raw = pd.read_parquet(PARQUET_RAW)[['nct_id', 'eligibility_criteria']]

if RERUN_ONLY:
    # Solo los llm_partial con ambas listas vacías del output existente
    df_prev = pd.read_parquet(PARQUET_OUTPUT)
    mask_rerun = (
        (df_prev['parse_status'] == 'llm_partial')
        & (df_prev['n_inclusion'] == 0)
        & (df_prev['n_exclusion'] == 0)
    )
    rerun_ids     = set(df_prev.loc[mask_rerun, 'nct_id'])
    df_to_process = df_raw[df_raw['nct_id'].isin(rerun_ids)].copy()
    df_to_process['task'] = 'full'
    df_regex        = None
    mask_to_process = None
    print(f'Modo RERUN_ONLY: {len(df_to_process):,} llm_partial vacíos a re-procesar')
else:
    # Pipeline completo desde regex
    df_regex = pd.read_parquet(PARQUET_REGEX)

    mask_pending         = df_regex['parse_status'] == 'llm_pending'
    mask_partial_no_excl = (df_regex['parse_status'] == 'partial') & (df_regex['n_exclusion'] == 0)
    mask_partial_no_incl = (df_regex['parse_status'] == 'partial') & (df_regex['n_inclusion'] == 0)
    mask_to_process      = mask_pending | mask_partial_no_excl | mask_partial_no_incl

    df_to_process = (
        df_regex[mask_to_process]
        .merge(df_raw, on='nct_id', how='left')
        .copy()
    )
    df_to_process['task'] = 'full'
    df_to_process.loc[mask_partial_no_excl[mask_to_process].values, 'task'] = 'missing_exclusion'
    df_to_process.loc[mask_partial_no_incl[mask_to_process].values, 'task'] = 'missing_inclusion'

    # Re-run adicional de llm_partial vacíos si ya existe output previo
    if PARQUET_OUTPUT.exists():
        df_prev = pd.read_parquet(PARQUET_OUTPUT)
        mask_rerun = (
            (df_prev['parse_status'] == 'llm_partial')
            & (df_prev['n_inclusion'] == 0)
            & (df_prev['n_exclusion'] == 0)
        )
        rerun_ids = set(df_prev.loc[mask_rerun, 'nct_id'])
        if rerun_ids:
            df_rerun = df_raw[df_raw['nct_id'].isin(rerun_ids)].copy()
            df_rerun['task'] = 'full'
            already  = set(df_to_process['nct_id'])
            df_rerun = df_rerun[~df_rerun['nct_id'].isin(already)]
            df_to_process = pd.concat([df_to_process, df_rerun], ignore_index=True)
            print(f'Re-run de llm_partial vacíos: {len(rerun_ids):,} casos adicionales')

    df_to_process = df_to_process.sort_values(
        'task', key=lambda x: x.map(TASK_ORDER)
    ).reset_index(drop=True)

# Resume desde checkpoint si existe
df_checkpoint = None
if PARQUET_CHECKPOINT.exists():
    df_checkpoint = pd.read_parquet(PARQUET_CHECKPOINT)
    done_ids      = set(df_checkpoint['nct_id'])
    df_to_process = df_to_process[~df_to_process['nct_id'].isin(done_ids)].reset_index(drop=True)
    print(f'Retomando desde checkpoint: {len(done_ids):,} ya procesados')

print(f"llm_pending (full):        {(df_to_process['task'] == 'full').sum():>6,}")
print(f"partial → falta exclusión: {(df_to_process['task'] == 'missing_exclusion').sum():>6,}")
print(f"partial → falta inclusión: {(df_to_process['task'] == 'missing_inclusion').sum():>6,}")
print(f"Total a procesar:          {len(df_to_process):>6,}")

Modo RERUN_ONLY: 3,742 llm_partial vacíos a re-procesar
llm_pending (full):         3,742
partial → falta exclusión:      0
partial → falta inclusión:      0
Total a procesar:           3,742


## 4.3 Setup Ollama

Requiere Ollama corriendo localmente. Si no está disponible:
```bash
ollama serve
ollama pull llama3.2:3b-instruct-q4_K_M
```

In [20]:
def check_ollama() -> bool:
    """Verifica que Ollama esté corriendo y el modelo esté disponible."""
    response = requests.get(f'{OLLAMA_URL}/api/tags', timeout=5)
    response.raise_for_status()
    available = [m['name'] for m in response.json().get('models', [])]
    print(f'Modelos disponibles: {available}')
    if MODEL not in available:
        print(f'⚠ Modelo {MODEL} no encontrado. Ejecutar: ollama pull {MODEL}')
        return False
    print(f'✓ {MODEL} listo')
    return True


check_ollama()

Modelos disponibles: ['llama3.2:3b-instruct-q4_K_M', 'mistral:7b-instruct-q4_K_M']
✓ llama3.2:3b-instruct-q4_K_M listo


True

## 4.4 Prompt y extracción

In [21]:
def needs_llm(text: str, missing_section: str) -> bool:
    """Detecta si el texto tiene señales de la sección faltante antes de llamar al LLM."""
    if missing_section == 'exclusion':
        return bool(_EXCL_SIGNALS.search(text))
    if missing_section == 'inclusion':
        return bool(_INCL_SIGNALS.search(text))
    return True


def sanitize_str(s: str) -> str:
    """Elimina surrogates inválidos que PyArrow no puede serializar."""
    return s.encode('utf-8', errors='ignore').decode('utf-8')


def sanitize_list(lst: list) -> list[str]:
    """Filtra y sanitiza una lista de criterios."""
    return [sanitize_str(c) for c in lst if isinstance(c, str) and c.strip()]


def build_prompt(text: str, missing_section: str | None = None) -> str:
    """Construye el prompt según si es extracción completa o de sección faltante."""
    base = """You are a clinical trial eligibility criteria extractor specializing in oncology trials.
The trials are oncology-focused, so criteria may reference cancer types, tumor stages, prior chemotherapy,
biomarkers, performance status (ECOG/Karnofsky), and oncology-specific treatments.

Rules:
- Extract criteria in their ORIGINAL language (text may be in English, Italian, German, or Spanish).
- Do NOT paraphrase, normalize, or rewrite criteria. Copy the text exactly as it appears.
- Split compound criteria joined by AND into separate items (e.g. \"age > 18 AND ECOG 0-2\" → two items).
- Preserve parenthetical notes and exceptions within the item they belong to.
- When a criterion is ambiguous or could belong to either list, classify it as INCLUSION.
"""
    if missing_section is None:
        task = """Your task: extract ALL inclusion and exclusion criteria from the text.
Respond ONLY with valid JSON, no other text:
{\"inclusion\": [\"criterion 1\", \"criterion 2\"], \"exclusion\": [\"criterion 1\", \"criterion 2\"]}
If a section is absent, use an empty list."""
    else:
        other = 'exclusion' if missing_section == 'inclusion' else 'inclusion'
        task = f"""Your task: the {other} criteria were already extracted. Extract ONLY the {missing_section} criteria.
IMPORTANT: if no {missing_section} criteria are explicitly present in the text, return an empty list.
Do NOT infer, negate, or invent criteria. Only extract text that explicitly appears in the input.
Respond ONLY with valid JSON, no other text:
{{\"{missing_section}\": [\"criterion 1\", \"criterion 2\"]}}
If no {missing_section} criteria are present, use an empty list."""

    return f"{base}\n{task}\n\nEligibility criteria text:\n{text}"


def call_ollama(prompt: str) -> str:
    """Llama al modelo y retorna el texto generado."""
    payload = {
        'model':   MODEL,
        'prompt':  prompt,
        'format':  'json',
        'stream':  False,
        'options': {'temperature': 0.0, 'num_ctx': 2048, 'num_predict': MAX_TOKENS},
    }
    response = requests.post(
        f'{OLLAMA_URL}/api/generate',
        json=payload,
        timeout=REQUEST_TIMEOUT
    )
    response.raise_for_status()
    return response.json()['response']


def parse_llm_response(raw: str, missing_section: str | None) -> dict | None:
    """Parsea la respuesta JSON; retorna None si está malformada o faltan keys."""
    try:
        data = json.loads(raw)
        if missing_section is None:
            if 'inclusion' in data and 'exclusion' in data:
                return data
        else:
            if missing_section in data:
                return data
        return None
    except json.JSONDecodeError:
        return None


def extract_criteria(text: str, missing_section: str | None = None) -> dict:
    """Extrae criterios con detección previa y reintentos ante JSON malformado."""
    if missing_section is not None and not needs_llm(text, missing_section):
        return {
            missing_section + '_criteria': [],
            'llm_ok': True,
        }

    prompt = build_prompt(text, missing_section)
    for _ in range(MAX_RETRIES):
        try:
            raw    = call_ollama(prompt)
            parsed = parse_llm_response(raw, missing_section)
            if parsed is not None:
                if missing_section is None:
                    return {
                        'inclusion_criteria': sanitize_list(parsed.get('inclusion', [])),
                        'exclusion_criteria': sanitize_list(parsed.get('exclusion', [])),
                        'llm_ok': True,
                    }
                else:
                    return {
                        missing_section + '_criteria': sanitize_list(parsed.get(missing_section, [])),
                        'llm_ok': True,
                    }
        except requests.exceptions.Timeout:
            pass
        time.sleep(0.5)

    return {'llm_ok': False}

In [22]:
# Smoke test — un caso de cada tipo presente
for task_type in ['full', 'missing_exclusion', 'missing_inclusion']:
    sample = df_to_process[df_to_process['task'] == task_type]
    if sample.empty:
        continue
    row     = sample.iloc[0]
    text    = row['eligibility_criteria'] or ''
    missing = None if task_type == 'full' else task_type.replace('missing_', '')
    t0      = time.time()
    result  = extract_criteria(text, missing)
    print(f'--- {task_type} | {time.time()-t0:.1f}s ---')
    print(f'Texto: {text[:200]}')
    print(f'Resultado: {json.dumps(result, indent=2)[:400]}')
    print()

--- full | 5.3s ---
Texto: DISEASE CHARACTERISTICS:

* Histologically confirmed breast cancer tumor samples collected on clinical trial ECOG-E2197
* Histologically lymph node-positive (N1-3) OR high-risk lymph node-negative dis
Resultado: {
  "inclusion_criteria": [
    "Histologically confirmed breast cancer tumor samples collected on clinical trial ECOG-E2197",
    "Histologically lymph node-positive (N1-3) OR high-risk lymph node-negative disease",
    "Hormone receptor-positive or negative disease (status known)",
    "HER2 status known",
    "Menopausal status not specified"
  ],
  "exclusion_criteria": [],
  "llm_ok": true
}



## 4.5 Pipeline (paralelo)

In [23]:
def process_row(args: tuple) -> dict:
    """Procesa una fila: llama al LLM y combina con lo que ya tenía el regex."""
    i, row = args
    text    = row['eligibility_criteria'] or ''
    task    = row['task']
    missing = None if task == 'full' else task.replace('missing_', '')

    extracted = extract_criteria(text, missing)
    extracted['nct_id'] = row['nct_id']
    extracted['task']   = task

    if task == 'missing_exclusion' and extracted.get('llm_ok'):
        extracted['inclusion_criteria'] = sanitize_list(list(row['inclusion_criteria']))
    elif task == 'missing_inclusion' and extracted.get('llm_ok'):
        extracted['exclusion_criteria'] = sanitize_list(list(row['exclusion_criteria']))

    return extracted


def save_checkpoint(completed_results: list) -> None:
    """Guarda los resultados completados hasta ahora al parquet de checkpoint."""
    df_partial = pd.DataFrame([r for r in completed_results if r is not None])
    if df_checkpoint is not None:
        df_partial = pd.concat([df_checkpoint, df_partial], ignore_index=True)
    df_partial.to_parquet(PARQUET_CHECKPOINT, index=False)


rows      = [(i, row) for i, (_, row) in enumerate(df_to_process.iterrows())]
total     = len(rows)
results   = [None] * total
completed = 0

print(f'Iniciando pipeline: {total} casos con {N_WORKERS} workers', flush=True)

with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
    futures = {executor.submit(process_row, r): r[0] for r in rows}
    for future in as_completed(futures):
        idx          = futures[future]
        results[idx] = future.result()
        completed   += 1

        if completed % PROGRESS_EVERY == 0 or completed == total:
            print(f'[{completed}/{total}] procesados', flush=True)

        if completed % CHECKPOINT_EVERY == 0:
            save_checkpoint(results)
            print(f'  checkpoint guardado ({completed} casos)', flush=True)

df_llm = pd.DataFrame(results)
print(f'\nLLM ok:     {df_llm["llm_ok"].sum():,}')
print(f'LLM failed: {(~df_llm["llm_ok"]).sum():,}')

Iniciando pipeline: 3742 casos con 4 workers
[25/3742] procesados
[50/3742] procesados
[75/3742] procesados
[100/3742] procesados
  checkpoint guardado (100 casos)
[125/3742] procesados
[150/3742] procesados
[175/3742] procesados
[200/3742] procesados
  checkpoint guardado (200 casos)
[225/3742] procesados
[250/3742] procesados
[275/3742] procesados
[300/3742] procesados
  checkpoint guardado (300 casos)
[325/3742] procesados
[350/3742] procesados
[375/3742] procesados
[400/3742] procesados
  checkpoint guardado (400 casos)
[425/3742] procesados
[450/3742] procesados
[475/3742] procesados
[500/3742] procesados
  checkpoint guardado (500 casos)
[525/3742] procesados
[550/3742] procesados
[575/3742] procesados
[600/3742] procesados
  checkpoint guardado (600 casos)
[625/3742] procesados
[650/3742] procesados
[675/3742] procesados
[700/3742] procesados
  checkpoint guardado (700 casos)
[725/3742] procesados
[750/3742] procesados
[775/3742] procesados
[800/3742] procesados
  checkpoint gua

## 4.6 Asignación de parse_status

In [24]:
def assign_llm_status(row: pd.Series) -> str:
    """Asigna parse_status según tipo de tarea y resultado."""
    if not row['llm_ok']:
        return 'llm_failed'
    if row['task'] in ('missing_exclusion', 'missing_inclusion'):
        return 'llm_enriched'
    if len(row.get('inclusion_criteria') or []) >= 2 and len(row.get('exclusion_criteria') or []) >= 2:
        return 'llm_clean'
    return 'llm_partial'


if df_checkpoint is not None:
    df_llm = pd.concat([df_checkpoint, df_llm], ignore_index=True)

df_llm['inclusion_criteria'] = df_llm['inclusion_criteria'].apply(
    lambda x: x if isinstance(x, list) else []
)
df_llm['exclusion_criteria'] = df_llm['exclusion_criteria'].apply(
    lambda x: x if isinstance(x, list) else []
)

df_llm['parse_status'] = df_llm.apply(assign_llm_status, axis=1)
df_llm['n_inclusion']  = df_llm['inclusion_criteria'].apply(len)
df_llm['n_exclusion']  = df_llm['exclusion_criteria'].apply(len)

print(df_llm['parse_status'].value_counts())

parse_status
llm_partial    2802
llm_clean       910
llm_failed       30
Name: count, dtype: int64


## 4.7 Merge y output

In [25]:
OUTPUT_COLS = ['nct_id', 'inclusion_criteria', 'exclusion_criteria', 'parse_status', 'n_inclusion', 'n_exclusion']

if RERUN_ONLY:
    # Solo actualizamos las filas re-procesadas en el output existente
    df_llm_ok   = df_llm[df_llm['llm_ok']][OUTPUT_COLS].copy()
    df_prev     = pd.read_parquet(PARQUET_OUTPUT)
    updated_ids = set(df_llm_ok['nct_id'])
    df_output   = pd.concat(
        [df_prev[~df_prev['nct_id'].isin(updated_ids)], df_llm_ok],
        ignore_index=True
    )
else:
    df_untouched = df_regex[~mask_to_process][OUTPUT_COLS].copy()
    df_llm_ok    = df_llm[df_llm['llm_ok']][OUTPUT_COLS].copy()

    failed_ids         = df_llm[~df_llm['llm_ok']]['nct_id'].tolist()
    df_failed_fallback = df_regex[df_regex['nct_id'].isin(failed_ids)][OUTPUT_COLS].copy()
    df_failed_fallback['parse_status'] = 'llm_failed'

    df_new = pd.concat([df_untouched, df_llm_ok, df_failed_fallback], ignore_index=True)

    if PARQUET_OUTPUT.exists():
        df_prev     = pd.read_parquet(PARQUET_OUTPUT)
        updated_ids = set(df_new['nct_id'])
        df_prev     = df_prev[~df_prev['nct_id'].isin(updated_ids)]
        df_output   = pd.concat([df_prev, df_new], ignore_index=True)
        print(f'Filas previas conservadas: {len(df_prev):,}')
    else:
        df_output = df_new

df_output.to_parquet(PARQUET_OUTPUT, index=False)

if PARQUET_CHECKPOINT.exists():
    PARQUET_CHECKPOINT.unlink()
    print('Checkpoint eliminado.')

print(f'Guardado: {PARQUET_OUTPUT}')
print(f'Filas totales: {len(df_output):,}')
print('\nDistribución parse_status final:')
print(df_output['parse_status'].value_counts())

Checkpoint eliminado.
Guardado: ..\data\04_eligibility_parsed_llm.parquet
Filas totales: 83,962

Distribución parse_status final:
parse_status
clean           64848
partial         10672
llm_partial      3592
llm_enriched     3319
llm_clean        1181
llm_failed        350
Name: count, dtype: int64


In [28]:
excel_path =  Path('..') / 'data' /'04_eligibility_parsed_llm.xlsx'

df_export = (
    df_output[df_output['parse_status'].str.startswith('llm_')]
    .merge(df_raw[['nct_id', 'eligibility_criteria']], on='nct_id', how='left')
    [['nct_id', 'inclusion_criteria', 'exclusion_criteria', 'eligibility_criteria']]
    .rename(columns={'nct_id': 'id'})
)

df_export.to_excel(excel_path, index=False)
print(f'Exportado: {excel_path} ({len(df_export):,} filas)')

Exportado: ..\data\04_eligibility_parsed_llm.xlsx (8,442 filas)
